In [52]:
import xarray as xr
import numpy as np
import dask
import socket
import pandas as pd
import glob
import xesmf as xe

from rossby_utils import cesm_utils as cesm
from functools import partial
import os

import importlib
importlib.reload(cesm)

import warnings
warnings.filterwarnings('ignore')

### Set up the dask cluster

In [6]:
from dask_jobqueue import PBSCluster
from dask.distributed import Client
dask.config.set({"distributed.scheduler.worker_saturation":1.0})
dask.config.set({"optimization.fuse.active": False})
dask.config.set({
    "distributed.worker.memory.target": 0.6,
    "distributed.worker.memory.spill": 0.7,
    "distributed.worker.memory.pause": 0.8,
    "distributed.worker.memory.terminate": 0.95,
})

cluster = PBSCluster(
    cores = 1,
    memory = '30GB',
    processes = 1,
    queue = 'casper',
    local_directory = '/glade/derecho/scratch/islas/dask_tmp/',
    resource_spec = 'select=1:ncpus=1:mem=30GB',
    project='P04010022',
    walltime='05:00:00',
    interface='mgt')

# scale up
cluster.scale(12)
#cluster.adapt(minimum=1, maximum=12)

# change your urls to the dask dashboard so that you can see it
hostname = socket.getfqdn()
dask.config.set({"distributed.dashboard.link":
        f"https://ondemand.hpc.ucar.edu/rnode/{hostname}/{{port}}/status"
})

client = Client(cluster)

/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/dask_jobqueue/core.py:266: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/miniforge/envs/npl-2026a/lib/python3.13/site-packages/dask_jobqueue/pbs.py:82: FutureWarning: project has been renamed to account as this kwarg was used wit -A option. You are still using it (please also check config files). If you did not set account yet, project will be respected for now, but it will be removed in a future release. If you already set account, project is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/miniforge/envs/npl-2026a/l

In [7]:
cluster

Dashboard: https://ondemand.hpc.ucar.edu/rnode/crhtc65.hpc.ucar.edu/40259/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.18.206.66:35397,Workers: 0
Dashboard: https://ondemand.hpc.ucar.edu/rnode/crhtc65.hpc.ucar.edu/40259/status,Total threads: 0
Started: Just now,Total memory: 0 B


### Set up info on data to be processed

In [58]:
ystart=1993
yend=2022 # !!! Note, SMYLE only goes to 2022
basepath="/glade/campaign/collections/gdex/data/d651065/SMYLE/archive/"
outpath="/glade/campaign/cgd/cas/islas/python_savs/rossbypalooza26/processing/CESM2/"
initmon=11
nmems=20
expname="b.e21.BSMYLE.f09_g17."
expnameXT="b.e21.BSMYLE-XT.f09_g17."
cesmvar='PSL' # naming convention for CESM
ourvar='slp' # naming convention we're using 

os.makedirs(outpath, exist_ok=True)

### Output grid

In [11]:
grid_out = xr.Dataset({'lat':(['lat'], np.arange(-90,90,1))}, {'lon': (['lon'], np.arange(0,360,1))})

### Grab the data, interpolate to the common 1 degree grid etc

In [25]:
def preprocessor(ds):
    # Fixing small round-off level differences in coordinates
    ds = cesm.round_lons_and_lats(ds)
    # Fix the CESM time axis
    ds = cesm.fix_cesm_time(ds)
    return ds

In [ ]:
reusewgt=False
wgtfile=outpath+'wgtfile.nc'
alltimes=[]
alldat=[]
for iyear in range(ystart,yend+1,1):
    if (iyear >= 2020):
        expname = expnameXT
    print(iyear)
    hindcast_months = pd.to_datetime([
        f"{iyear}-11-15",
        f"{iyear}-12-15",
        f"{iyear+1}-01-15",
        f"{iyear+1}-02-15",
        f"{iyear+1}-03-15"])

    allmonths = []

    # get the filelist for all November initializations for this year
    filelist = [ 
      glob.glob(basepath+expname+str(iyear)+'-'+str(initmon).zfill(2)+'.'+str(imem).zfill(3)+'/atm/proc/tseries/month_1/*.cam.h0.PSL.*.nc')
        for imem in np.arange(1,nmems+1,1) ]
    dat = xr.open_mfdataset(filelist, concat_dim=['member','time'], combine='nested', preprocess = partial(preprocessor))
    dat = dat[cesmvar]
    dat = dat.chunk({"member":20})

    # select out November to March
    dat = dat.sel(time=slice(str(iyear)+'-11-01',str(iyear+1)+'-03-31'))

    # regrid
    if (iyear == ystart):
        regridder = xe.Regridder(dat, grid_out, 'bilinear', periodic=True, reuse_weights=reusewgt, filename=wgtfile)
    dat_rg = regridder(dat)

    # turn time into a lead axis
    dat_rg = dat_rg.rename({'time':'lead'})
    dat_rg['lead'] = np.arange(1,5+1,1)

    # comupte and append
    dat_rg = dat_rg.compute()
    alldat.append(dat_rg)
    
    times = xr.DataArray(hindcast_months, dims=['lead'], coords=[np.arange(1,5+1,1)], name='time')
    alltimes.append(times)

alldat = xr.concat(alldat, dim='year')
alldat['year'] = range(ystart,yend+1,1)
alldat = alldat.rename(ourvar)
alltimes = xr.concat(alltimes, dim='year')
alltimes['year'] = range(ystart,yend+1,1)

datout = xr.merge([alldat, alltimes])
datout.to_netcdf(outpath+'CESM2_SMYLE_JRA_L32'+ourvar+'_'+str(ystart)+'_'+str(yend)+'.nc')

In [65]:
cluster.close()